**Raw Inspection**

In [38]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [39]:
# 1. Load the raw MTA Subway Station data
# Adjust the filename if yours is different (e.g., 'MTA_Subway_Stations.csv')
df_subway_raw = pd.read_csv('../data/mta_subway/MTA_Subway_Stations.csv')

# 2. Check the size and column names
print(f"Total subway stations found: {len(df_subway_raw)}")
print("\n--- Column Names ---")
print(df_subway_raw.columns.tolist())

# 3. Look at the first few rows to understand the structure
print("\n--- Raw Data Sample ---")
print(df_subway_raw[['Station ID', 'Complex ID', 'Stop Name', 'GTFS Latitude', 'GTFS Longitude']].head())

Total subway stations found: 496

--- Column Names ---
['GTFS Stop ID', 'Station ID', 'Complex ID', 'Division', 'Line', 'Stop Name', 'Borough', 'CBD', 'Daytime Routes', 'Structure', 'GTFS Latitude', 'GTFS Longitude', 'North Direction Label', 'South Direction Label', 'ADA', 'ADA Northbound', 'ADA Southbound', 'ADA Notes', 'Georeference']

--- Raw Data Sample ---
   Station ID  Complex ID             Stop Name  GTFS Latitude  GTFS Longitude
0           1           1  Astoria-Ditmars Blvd      40.775036      -73.912034
1           2           2          Astoria Blvd      40.770258      -73.917843
2           3           3                 30 Av      40.766779      -73.921479
3           4           4              Broadway      40.761820      -73.925508
4           5           5                 36 Av      40.756804      -73.929575


**Quality & Integrity Check**

In [40]:
# 1. Check for missing coordinates
null_coords = df_subway_raw[['GTFS Latitude', 'GTFS Longitude']].isnull().sum()
print(f"\nMissing Latitudes: {null_coords['GTFS Latitude']}")
print(f"Missing Longitudes: {null_coords['GTFS Longitude']}")

# 2. Check Coordinate Ranges (NYC Validation)
# Latitude should be ~40 and Longitude ~-73/-74
print("\n--- Coordinate Range Check ---")
print(df_subway_raw[['GTFS Latitude', 'GTFS Longitude']].describe())


Missing Latitudes: 0
Missing Longitudes: 0

--- Coordinate Range Check ---
       GTFS Latitude  GTFS Longitude
count     496.000000      496.000000
mean       40.723205      -73.945948
std         0.082440        0.069813
min        40.512764      -74.251961
25%        40.672557      -73.986957
50%        40.715021      -73.950960
75%        40.773485      -73.903092
max        40.903125      -73.755405


In [41]:
# 2. Extract the 263 unique zones and their coordinates
# We use the X and Y columns we saved earlier to rebuild the points
zones_unique = df_backbone[['PULocationID', 'centroid_x', 'centroid_y']].drop_duplicates()

# 3. Create the 'geometry' column from X and Y
zones_unique['geometry'] = zones_unique.apply(lambda r: Point(r.centroid_x, r.centroid_y), axis=1)

# 4. Turn it into a GeoDataFrame (defining it as EPSG:2263)
zones_unique_gdf = gpd.GeoDataFrame(zones_unique, geometry='geometry', crs="EPSG:2263")

print(f"Variable 'zones_unique_gdf' is now defined with {len(zones_unique_gdf)} unique zones.")

Variable 'zones_unique_gdf' is now defined with 263 unique zones.


**Feature Selection**

In [42]:
# Keep only what we need for the spatial join
subway_cols = ['Station ID', 'Stop Name', 'GTFS Latitude', 'GTFS Longitude']
df_subway_clean = df_subway_raw[subway_cols].copy()

print(f"\nCleaned Subway Data Shape: {df_subway_clean.dtypes}")


Cleaned Subway Data Shape: Station ID          int64
Stop Name          object
GTFS Latitude     float64
GTFS Longitude    float64
dtype: object


**Geometric Transformation**

In [49]:
import geopandas as gpd
from shapely.geometry import Point

# Create the GeoDataFrame
# We use EPSG:4326 because raw Lat/Lon is always in degrees
subway_gdf = gpd.GeoDataFrame(
    df_subway_clean, 
    geometry=gpd.points_from_xy(df_subway_clean['GTFS Longitude'], df_subway_clean['GTFS Latitude']),
    crs="EPSG:4326"
)

subway_gdf

,Station ID,Stop Name,GTFS Latitude,GTFS Longitude,geometry
0,1,Astoria-Ditmars Blvd,40.775036,-73.912034,POINT (-73.91203 40.77504)
1,2,Astoria Blvd,40.770258,-73.917843,POINT (-73.91784 40.77026)
2,3,30 Av,40.766779,-73.921479,POINT (-73.92148 40.76678)
3,4,Broadway,40.761820,-73.925508,POINT (-73.92551 40.76182)
4,5,36 Av,40.756804,-73.929575,POINT (-73.92958 40.7568)
...,...,...,...,...,...
491,517,Prince's Bay,40.525507,-74.200064,POINT (-74.20006 40.52551)
492,518,Pleasant Plains,40.522410,-74.217847,POINT (-74.21785 40.52241)
493,519,Richmond Valley,40.519631,-74.229141,POINT (-74.22914 40.51963)
494,522,Tottenville,40.512764,-74.251961,POINT (-74.25196 40.51276)


**Coordinate System Alignment**

In [44]:
# Project to NYC State Plane (Feet)
subway_gdf = subway_gdf.to_crs(epsg=2263)

print(f"Subway CRS aligned to Taxi Zones: {subway_gdf.crs}")

Subway CRS aligned to Taxi Zones: EPSG:2263


**"Nearest Neighbor" Calculation**

In [45]:
# We use the unique zones list we prepared earlier
# 1. Calculate the distance (result is in feet)
zones_unique_gdf['dist_to_subway_feet'] = zones_unique_gdf.geometry.apply(
    lambda x: subway_gdf.distance(x).min()
)

# 2. Convert to Kilometers (Methodology requirement)
# Conversion: Feet / 3280.84 = Kilometers
zones_unique_gdf['subway_proximity_km'] = zones_unique_gdf['dist_to_subway_feet'] / 3280.84

print("Feature Engineering: Subway Proximity created!")

Feature Engineering: Subway Proximity created!


**Merging Data**

In [46]:
# 1. Merge the proximity feature from your lookup table (zones_unique_gdf) 
# back into your main backbone (df_backbone)
df_final = pd.merge(
    df_backbone, 
    zones_unique_gdf[['PULocationID', 'subway_proximity_km']], 
    on='PULocationID', 
    how='left'
)

# 2. Verify the merge
print(f"Merged Dataset Shape: {df_final.shape}")
print(df_final[['PULocationID', 'zone', 'subway_proximity_km']].head())

Merged Dataset Shape: (568080, 11)
   PULocationID            zone  subway_proximity_km
0             1  Newark Airport            10.025964
1             1  Newark Airport            10.025964
2             1  Newark Airport            10.025964
3             1  Newark Airport            10.025964
4             1  Newark Airport            10.025964


In [47]:
# Check all column names to see the 'marriage' of taxi and subway data
print("--- List of all columns in the Combined Dataset ---")
print(df_final.columns.tolist())

# Look at a row where there actually IS demand (not just Newark 0s)
# This will show you how demand and subway proximity sit together
print("\n--- Verification of Integration (Rows with Pickups) ---")
print(df_final[df_final['pickup_count'] > 0].head())

--- List of all columns in the Combined Dataset ---
['PULocationID', 'date', 'hour', 'day_of_week', 'month', 'pickup_count', 'zone', 'borough', 'centroid_x', 'centroid_y', 'subway_proximity_km']

--- Verification of Integration (Rows with Pickups) ---
    PULocationID        date  hour  day_of_week  month  pickup_count  \
6              1  2025-01-01     6            2      1             1   
13             1  2025-01-01    13            2      1             1   
14             1  2025-01-01    14            2      1             1   
16             1  2025-01-01    16            2      1             1   
18             1  2025-01-01    18            2      1             1   

              zone borough     centroid_x     centroid_y  subway_proximity_km  
6   Newark Airport     EWR  935996.821016  191376.749531            10.025964  
13  Newark Airport     EWR  935996.821016  191376.749531            10.025964  
14  Newark Airport     EWR  935996.821016  191376.749531            10.0259

**Data Exportation**

In [48]:
# 1. Clean up temporary columns
# We keep subway_proximity_km, but drop the raw feet version
if 'dist_to_subway_feet' in df_final.columns:
    df_final = df_final.drop(columns=['dist_to_subway_feet'])

# 2. Export as the new checkpoint
# This file now has 568,080 rows and 10+ columns of clean, enriched data
df_final.to_parquet(
    '../data/processed/taxi_demand_enriched_v1.parquet', 
    index=False, 
    engine='fastparquet'
)

print("Export Complete: 'taxi_demand_enriched_v1.parquet' is ready.")

Export Complete: 'taxi_demand_enriched_v1.parquet' is ready.
